### Imports

In [21]:
# Import des outils de tables
import pandas as pd
import kagglehub

# Import des outils de machine learning
from sklearn.model_selection import train_test_split

# Import des modèles
from tabicl import TabICLRegressor
from sklearn.ensemble import RandomForestRegressor

# Import des métriques d'évaluation
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

### Téléchargement du dataset

In [22]:
# Exemple :
df = kagglehub.dataset_load(
    kagglehub.KaggleDatasetAdapter.PANDAS,
    "mirichoi0218/insurance",
    "insurance.csv"
)

df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.924
1,18,male,33.770,1,no,southeast,1725.552
2,28,male,33.000,3,no,southeast,4449.462
3,33,male,22.705,0,no,northwest,21984.471
4,32,male,28.880,0,no,northwest,3866.855


### Séparation en train, test

In [ ]:
# Variables explicatives
X = df.iloc[:, :-1]

# Conversion des variables catégorielles en variables numériques (One-Hot Encoding)
X = pd.get_dummies(X, drop_first=True)

# Variable à prédire
y = df.iloc[:, -1]  # df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(1070, 8)
(268, 8)
(1070,)
(268,)


### Prédiction avec TFM (Tabular Foundation Model)

In [24]:
model_tfm = TabICLRegressor()
model_tfm.fit(X_train, y_train)

y_pred_tfm = model_tfm.predict(X_test)

### Prédiction avec Random Forest

In [25]:
model_rf = RandomForestRegressor(
    random_state=42
)

model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)

### Évaluation des résultats

In [26]:
# ------------------------------------------------------------
# Métriques du TFM (Tabular Foundation Model - TabICL)
# Prédictions stockées dans : y_pred_tfm
# ------------------------------------------------------------

mae_tfm = mean_absolute_error(y_test, y_pred_tfm)

mse_tfm = mean_squared_error(y_test, y_pred_tfm)

rmse_tfm = np.sqrt(mse_tfm)

r2_tfm = r2_score(y_test, y_pred_tfm)


# ------------------------------------------------------------
# Métriques du Random Forest
# Prédictions stockées dans : y_pred_rf
# ------------------------------------------------------------

mae_rf = mean_absolute_error(y_test, y_pred_rf)

mse_rf = mean_squared_error(y_test, y_pred_rf)

rmse_rf = np.sqrt(mse_rf)

r2_rf = r2_score(y_test, y_pred_rf)

### Tableau Comparatif

In [27]:
resultats = pd.DataFrame({

    "Métrique": [
        "MAE",
        "MSE",
        "RMSE",
        "R²"
    ],

    "TFM (TabICL)": [
        mae_tfm,
        mse_tfm,
        rmse_tfm,
        r2_tfm
    ],

    "Random Forest": [
        mae_rf,
        mse_rf,
        rmse_rf,
        r2_rf
    ],

    "Description": [
        "MAE : plus proche de 0 = meilleur",
        "MSE : plus proche de 0 = meilleur",
        "RMSE : plus proche de 0 = meilleur",
        "R² : plus proche de 1 = meilleur"
    ]
})


# Formate les scores pour faciliter la lecture
def format_nombre(x):
    if abs(x) >= 1:
        return f"{x:_.0f}".replace("_", " ")
    else:
        return f"{x:.3f}"

resultats["TFM (TabICL)"] = resultats["TFM (TabICL)"].apply(format_nombre)
resultats["Random Forest"] = resultats["Random Forest"].apply(format_nombre)

# Désactiver la notation scientifique
pd.options.display.float_format = '{:.3f}'.format

# Affichage du tableau comparatif
resultats

,Métrique,TFM (TabICL),Random Forest,Description
0,MAE,2 429,2 550,MAE : plus proche de 0 = meilleur
1,MSE,17 590 907,20 942 405,MSE : plus proche de 0 = meilleur
2,RMSE,4 194,4 576,RMSE : plus proche de 0 = meilleur
3,R²,0.887,0.865,R² : plus proche de 1 = meilleur
